# SatQuery AI — Stage 4: Optical-SAR Fusion (QLoRA)

**Base model:** `Qwen/Qwen2-VL-2B-Instruct` (2B, fits T4 ~15GB VRAM)  
**Method:** QLoRA — 4-bit NF4 base + LoRA adapters (PEFT)  
**Hardware:** Google Colab free-tier T4 (15GB VRAM). Fallback: Kaggle T4×2 (`/kaggle/working` instead of Drive).  
**Purpose:** Train a **dedicated** optical-SAR fusion LoRA that has actually seen real co-registered Sentinel-1 + Sentinel-2 pairs (replaces the reused VRSBench optical-only adapter currently used in `backend/models/fusion.py`).

> **Constraints from `AGENTS.md` — do not simplify away:**
> - QLoRA only (never full fine-tune), small VLM 2–3B only
> - Checkpoint to Drive every ~25 steps + auto-resume (safe to rerun top-to-bottom after disconnect)
> - Dataset subsets cached to Drive once, never re-download full BigEarthNet/GFM-Bench per session
> - `batch_size=1` + `gradient_accumulation=8` (fits T4), gradient checkpointing tuned for T4
> - Each stage has its own notebook + own checkpoint dir; stage 4 loads stage 3 adapter `imadityasarkar/cdvqa_change`

---
**How to use:** `Runtime → Run all` (or run cells top-to-bottom). After any disconnect, just run again — it resumes from the latest Drive checkpoint automatically.  
**Expected time:** ~15–25 min for the default 800-sample subset, 1 epoch, on T4.


## 1 — Verify GPU (T4, CUDA, VRAM)

If this shows **Tesla T4** and `torch.cuda.is_available() == True`, you are good. If no GPU: `Runtime → Change runtime type → T4 GPU`.


In [ ]:
import sys, platform, torch
!nvidia-smi  2>&1 | head -n 20
print(f"Python: {sys.version.split()[0]} | platform: {platform.platform()}")
print(f"Torch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"Compute capability: {props.major}.{props.minor} (T4 is 7.5)")
    print(f"Total VRAM: {props.total_memory/1024**3:.1f} GB")
    # T4 is sm_75, does NOT support bf16 efficiently — we will use fp16
    print(f"bf16 supported: {torch.cuda.is_bf16_supported() if hasattr(torch.cuda, 'is_bf16_supported') else 'unknown (assume False on T4)'}")
else:
    print("⚠️ No GPU detected — switch runtime to T4 before continuing.")
    # Do not raise — let user fix runtime


## 2 — Mount Google Drive (checkpoints + dataset cache)

All checkpoints and the cached subset live on Drive so training survives disconnects.

- `DRIVE_ROOT = /content/drive/MyDrive/SatQueryAI` (change if you prefer another path)
- `CHECKPOINT_DIR = DRIVE_ROOT/checkpoints/stage4_optical_sar_fusion`
- `DATASET_CACHE_DIR = DRIVE_ROOT/datasets/gfm_bench_optical_sar_subset`

On Kaggle, replace `/content/drive/MyDrive/...` with `/kaggle/working/SatQueryAI`.


In [ ]:
from pathlib import Path
import os

# --- CONFIGURE THESE IF NEEDED ---
DRIVE_ROOT = Path("/content/drive/MyDrive/SatQueryAI")
CHECKPOINT_DIR = DRIVE_ROOT / "checkpoints" / "stage4_optical_sar_fusion"
DATASET_CACHE_DIR = DRIVE_ROOT / "datasets" / "gfm_bench_optical_sar_subset"
ADAPTER_OUTPUT_DIR = CHECKPOINT_DIR / "final_adapter"

# Mount (no-op if already mounted)
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    print("Drive mounted.")
except ImportError:
    print("Not in Colab (e.g. local/Kaggle) — skipping drive.mount. Set DRIVE_ROOT accordingly.")

for p in [DRIVE_ROOT, CHECKPOINT_DIR, DATASET_CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
    print(f"{p} -> exists={p.exists()}")

print(f"\nCHECKPOINT_DIR: {CHECKPOINT_DIR}")
print(f"DATASET_CACHE_DIR: {DATASET_CACHE_DIR}")
# List existing checkpoints (if any)
!ls -lh "{CHECKPOINT_DIR}" 2>&1 | head -n 30


## 3 — Install dependencies (restart-free, T4-safe)

**Goal:** avoid the classic Colab breakage `libnvJitLink.so.13` / `CUDA SETUP ERROR` that happens when `pip install -U` upgrades `torch` from `cu128` → `cu130`.

- We **do not upgrade `torch`** — keep Colab's preinstalled `torch 2.10+cu128` (or 2.8+cu126). It already satisfies all deps.
- We pin `transformers>=4.46.0`, `peft`, `accelerate`, `bitsandbytes`, `datasets`, `qwen-vl-utils`, `trl` to mutually compatible wheels tested on T4.
- We force-reinstall `pillow` to avoid the `transformers>=4.46.0` vs `pillow<10` conflict this repo hit once already.
- If you see `ImportError: cannot import name 'sync_gpu'` after install, you forgot to restart — just `Runtime → Restart runtime` and re-run from here.

> **After this cell finishes, do `Runtime → Restart runtime` if the output says "Restart required" or if `import bitsandbytes` fails. Then re-run cells from §1 again (they are idempotent).**


In [ ]:
import torch
print(f"Before install — torch {torch.__version__} | cuda {torch.version.cuda if hasattr(torch.version,'cuda') else 'unknown'}")

# Install pinned, T4-tested versions WITHOUT upgrading torch/nvidia libs.
# These versions are known to work together on Colab Python 3.11, CUDA 12.8, T4 (sm_75).
# Pillow is force-reinstalled to avoid transformers>=4.46.0 conflict (repo hit this once).
!pip install -q \
    "transformers>=4.46.0" \
    "peft==0.14.0" \
    "accelerate==1.4.0" \
    "bitsandbytes==0.45.5" \
    "datasets==3.1.0" \
    "qwen-vl-utils==0.0.10" \
    "trl==0.15.2" \
    "numpy"
!pip install -q --force-reinstall "pillow>=10.0.0" 2>&1 | tail -n 5

print("\nInstall done. Checking imports (no restart needed if all import ok)...")
import importlib
for pkg in ["transformers","peft","accelerate","bitsandbytes","datasets","qwen_vl_utils","trl","PIL"]:
    try:
        m = importlib.import_module(pkg)
        print(f"  {pkg}: {getattr(m,'__version__','ok')}")
    except Exception as e:
        print(f"  {pkg}: FAILED — {e}")

# Quick bitsandbytes CUDA sanity check
!python -m bitsandbytes 2>&1 | head -n 30


### 3b — If you hit CUDA / bitsandbytes errors, run this cell then restart

Common symptoms and fixes:
- `CUDA SETUP ERROR: libnvJitLink.so.13 cannot open` → you upgraded to `torch+cu130` but Colab still has `cu12` libs. Fix: `pip uninstall nvidia-nvjitlink-cu12` or reinstall torch `cu128` (uncomment below).
- `cannot import name 'sync_gpu'` → version mix from upgrading without restart. Fix: restart runtime.
- `libcusparse.so.11` → bitsandbytes compiled for different CUDA. Fix: reinstall pinned `bitsandbytes==0.45.5` and restart.

Normally you can skip this cell. Only run if the previous cell's import check failed.


In [ ]:
# >>> SKIP THIS CELL unless you saw an error in §3 <<<
# Uncomment ONE of the fixes below if needed, then Runtime → Restart runtime

# Fix 1: clean mixed CUDA 12/13 jitlink (most common when torch was upgraded to cu130)
# !pip uninstall -y nvidia-nvjitlink-cu12 nvidia-nvjitlink-cu13 2>&1 | tail -n 5

# Fix 2: force torch back to Colab's default cu128 wheel (if you accidentally got cu130)
# !pip uninstall -y torch torchvision torchaudio 2>&1 | tail -n 5
# !pip install -q "torch==2.10.0" "torchvision==0.25.0" "torchaudio==2.10.0" --index-url https://download.pytorch.org/whl/cu128 2>&1 | tail -n 5

# Fix 3: reinstall bitsandbytes clean
# !pip install -q --force-reinstall "bitsandbytes==0.45.5" 2>&1 | tail -n 5

print("If you uncommented a fix, now do Runtime → Restart runtime, then re-run from §1.")


## 4 — Imports + version audit

Imports are isolated here so a restart clearly fixes any `bitsandbytes` mix. This cell must pass without error before training.


In [ ]:
import os, json, random, math, glob, warnings
from pathlib import Path
from PIL import Image
import numpy as np
import torch

import transformers, peft, accelerate, bitsandbytes, datasets, trl
print(f"transformers {transformers.__version__}")
print(f"peft {peft.__version__}")
print(f"accelerate {accelerate.__version__}")
print(f"bitsandbytes {bitsandbytes.__version__}")
print(f"datasets {datasets.__version__}")
print(f"trl {trl.__version__}")
print(f"torch {torch.__version__} | cuda {torch.version.cuda}")

# Must be on GPU for QLoRA
assert torch.cuda.is_available(), "CUDA not available — set Runtime → T4 GPU"
device_name = torch.cuda.get_device_name(0)
print(f"Using GPU: {device_name}")

# T4 is fp16-only for practical purposes (bf16 is slow/emulated)
COMPUTE_DTYPE = torch.float16  # keep fp16 on T4
USE_BF16 = False
print(f"Compute dtype: {COMPUTE_DTYPE} (fp16 on T4, bf16 would be slower)")


## 5 — Training config (single source of truth)

Edit these values to adapt the run. They are also saved to `training/configs/stage4_optical_sar_fusion.json` for reproducibility (mirrors `training/configs/` on disk).

- Stage 4 continues from **stage 3** `imadityasarkar/cdvqa_change` (latest in chain: stage1 BigEarthNet → stage2 VRSBench → stage3 CDVQA → stage4 Fusion). Change `adapter_to_continue` if you want to branch from a different stage.
- `sar_viz_mode` controls SAR false-color mapping (`vv_vh_ratio` default: VV→R, VH→G, VV/VH ratio→B).
- `SUBSET_SIZE` controls the cached subset size — start small (e.g. 500–800) for free Colab, scale up once caching works.


In [ ]:
from dataclasses import dataclass, asdict

@dataclass
class TrainConfig:
    # Model
    base_model: str = "Qwen/Qwen2-VL-2B-Instruct"
    # Stage 4 continues from stage 3 (latest in chain). Use HF id or Drive path /content/drive/MyDrive/SatQueryAI/checkpoints/stage3_cdvqa_change/final_adapter
    adapter_to_continue: str = "imadityasarkar/cdvqa_change"
    # SAR viz mode — false-color mapping for Sentinel-1 GRD
    sar_viz_mode: str = "vv_vh_ratio"  # vv→R, vh→G, vv/vh ratio→B, normalized from dB -25..0

    # Data — Optical-SAR fusion via GFM-Bench/BigEarthNet (same dataset as stage 1, now both modalities)
    subset_size: int = 800
    val_ratio: float = 0.05
    image_max_pixels: int = 384*28*28
    image_min_pixels: int = 192*28*28
    max_seq_length: int = 768

    # QLoRA
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.05
    lora_target_modules: tuple = ("q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj")

    # Optimization — T4-friendly
    per_device_train_batch_size: int = 1
    gradient_accumulation_steps: int = 8
    learning_rate: float = 1e-4  # SFT stage, lower than stage1 2e-4
    lr_scheduler_type: str = "cosine"
    warmup_ratio: float = 0.05
    num_train_epochs: int = 1
    max_steps: int = -1
    weight_decay: float = 0.01
    optim: str = "paged_adamw_8bit"
    gradient_checkpointing: bool = True
    max_grad_norm: float = 1.0

    # Checkpointing (frequent saves for Drive)
    save_steps: int = 25
    save_total_limit: int = 2
    logging_steps: int = 5
    eval_steps: int = 50
    seed: int = 3407

CFG = TrainConfig()
print(json.dumps(asdict(CFG), indent=2))

# Persist config to Drive + repo configs/
repo_config_path = Path("/content/drive/MyDrive/SatQueryAI/training_config_stage4_fusion.json")
try:
    repo_config_path.parent.mkdir(parents=True, exist_ok=True)
    repo_config_path.write_text(json.dumps(asdict(CFG), indent=2))
    print(f"Saved Drive config: {repo_config_path}")
except Exception as e:
    print(f"Could not save Drive config: {e}")
try:
    local_cfg = Path("training/configs/stage4_optical_sar_fusion.json")
    local_cfg.parent.mkdir(parents=True, exist_ok=True)
    local_cfg.write_text(json.dumps(asdict(CFG), indent=2))
    print(f"Saved local config: {local_cfg.resolve()}")
except Exception as e:
    print(f"Local save skipped: {e}")


## 6 — Dataset subset & cache (Drive)

`GFM-Bench/BigEarthNet` is ~70GB full; we never download it in one Colab session. Instead:

1. **Check `DATASET_CACHE_DIR` on Drive** — if a cached subset exists, load it with `load_from_disk` (instant, no download).
2. Otherwise **build a small subset** via streaming `GFM-Bench/BigEarthNet` with `streaming=True` + `shuffle(buffer_size=1000)` and **save to Drive** for next session. We pull **BOTH** `optical` (Sentinel-2, B04/B03/B02 → RGB) and `radar` (Sentinel-1, VV/VH → false-color) per sample, co-registered by construction.

The subset is stored as a HuggingFace `Dataset` with columns `optical_image` (PIL), `sar_image` (PIL), `question`, `answer`. Swap the synthetic fallback if needed — the rest of the notebook is loader-agnostic.

> Synthetic fallback generates random optical+SAR chips + templated captions so the notebook always runs end-to-end (useful for CI / first-time verification). For real training, the streaming builder above runs.


In [ ]:
from datasets import Dataset, DatasetDict, load_from_disk, load_dataset
from PIL import Image
import io, random, numpy as np

# -- Helpers reused from stage 1 (exact band selection) --
def optical_to_pil(optical) -> Image.Image:
    """Convert Sentinel-2 optical (B04,B03,B02 or stacked array) to RGB PIL.
    Reuses exact stage-1 function: B04→R, B03→G, B02→B, reflectance 0-10000 → 0-255 with clipping.
    Accepts: numpy array (H,W,3) or dict with bands, or already PIL.
    """
    if isinstance(optical, Image.Image):
        return optical.convert("RGB")
    # Handle dict with band keys if dataset returns separate bands
    if isinstance(optical, dict):
        # Try common keys
        for k in ["optical","image","B04","B4"]:
            if k in optical:
                optical = optical[k]
                break
    # If still dict, try to stack B04/B03/B02
    if isinstance(optical, dict) and "B04" in optical:
        try:
            b04 = np.array(optical["B04"])
            b03 = np.array(optical["B03"])
            b02 = np.array(optical["B02"])
            arr = np.stack([b04,b03,b02], axis=-1)
            optical = arr
        except Exception:
            pass
    arr = np.array(optical)
    # Squeeze if (C,H,W) -> (H,W,C)
    if arr.ndim==3 and arr.shape[0] in [3,4,12,13]:
        # Heuristic: if first dim is bands, transpose
        if arr.shape[0] <= 4 and arr.shape[1] > 10:
            arr = np.transpose(arr, (1,2,0))
    # If more than 3 bands, select B04,B03,B02 as indices 3,2,1 for Sentinel-2 L2A (B02=1,B03=2,B04=3 in 0-indexed 12-band stack)
    if arr.ndim==3 and arr.shape[-1] > 3:
        try:
            # Common BigEarthNet ordering: B02,B03,B04 are indices 1,2,3 (0-based) in 12-band set
            arr = arr[:,:, [3,2,1]]  # B04,B03,B02
        except Exception:
            arr = arr[:,:,:3]
    # Normalize reflectance 0-10000 or 0-1 to 0-255
    if arr.dtype != np.uint8:
        # Heuristic: if max > 1, assume 0-10000 scaling
        if arr.max() > 1.5:
            arr = np.clip(arr / 10000 * 255, 0, 255).astype(np.uint8)
        else:
            arr = np.clip(arr * 255, 0, 255).astype(np.uint8)
    if arr.shape[-1] != 3:
        arr = arr[:,:,:3]
    return Image.fromarray(arr).convert("RGB")

def sar_to_pil(radar, mode="vv_vh_ratio") -> Image.Image:
    """Convert Sentinel-1 radar (VV,VH) to false-color PIL.
    VV -> red, VH -> green, VV/VH ratio -> blue, each normalized/clipped to 0-255.
    Sentinel-1 GRD values are typically in dB roughly -25 to 0 — clip and normalize accordingly,
    not assuming the 0-10000 reflectance scaling used for optical.
    Accepts: numpy array (H,W,2) with VV,VH, or dict.
    """
    if isinstance(radar, Image.Image):
        return radar.convert("RGB")
    if isinstance(radar, dict):
        # Try keys VV/VH
        if "VV" in radar and "VH" in radar:
            try:
                vv = np.array(radar["VV"], dtype=np.float32)
                vh = np.array(radar["VH"], dtype=np.float32)
                radar = np.stack([vv,vh], axis=-1)
            except Exception:
                pass
        elif "radar" in radar:
            radar = radar["radar"]
    arr = np.array(radar)
    if arr.ndim==3 and arr.shape[0]==2 and arr.shape[1]>10:
        arr = np.transpose(arr, (1,2,0))
    if arr.ndim==2:
        arr = np.stack([arr, arr], axis=-1)
    if arr.ndim==3 and arr.shape[-1]>2:
        arr = arr[:,:,:2]
    # arr now H,W,2 with VV,VH in dB or linear
    vv = arr[:,:,0].astype(np.float32)
    vh = arr[:,:,1].astype(np.float32)
    # If values look like linear power (0-1) vs dB (-25..0), heuristic: if mean >1, assume linear then convert to dB
    # But we just clip dB range -25..0 for normalization
    # Normalize VV: -25..0 -> 0..255
    def norm_db(x):
        return np.clip((x + 25) / 25 * 255, 0, 255).astype(np.uint8)
    r = norm_db(vv)
    g = norm_db(vh)
    # Blue = VV/VH ratio in dB: VV - VH (since dB), range roughly -10..10 -> 0..255
    ratio = vv - vh  # dB ratio
    b = np.clip((ratio + 10) / 20 * 255, 0, 255).astype(np.uint8)
    if mode != "vv_vh_ratio":
        # Alternative modes could be added here
        pass
    rgb = np.stack([r,g,b], axis=-1)
    return Image.fromarray(rgb, mode="RGB")

LAND_COVER_LABELS = ["forest", "urban fabric", "arable land", "pasture", "water bodies", "industrial units", "shrubland", "wetlands", "inland waters", "continuous urban fabric"]

def label_to_caption(labels) -> str:
    """Reuse existing function — maps BigEarthNet label list to sentence."""
    if isinstance(labels, str):
        labels = [labels]
    # Join with natural phrasing
    if len(labels)==0:
        return "This Sentinel-2 image shows mixed land cover."
    if len(labels)==1:
        return f"This Sentinel-2 image shows predominantly {labels[0]}."
    return f"This Sentinel-2 image shows {', '.join(labels[:-1])} and {labels[-1]}."

FUSION_QUESTION = "Use the optical and SAR images together to identify built-up and water-covered regions."

WATER_LABELS = {"water bodies","inland waters","water", "wetlands", "marine waters"}
URBAN_LABELS = {"urban fabric","continuous urban fabric","discontinuous urban fabric","industrial units","built-up", "artificial surfaces"}

def build_answer_with_sar_heuristic(labels, base_caption: str) -> str:
    # Heuristic template for training signal — not a physically rigorous SAR interpretation claim.
    # Adds SAR-grounded sentence when label set suggests water/urban, to teach fusion.
    extra = ""
    labs = set([l.lower() for l in labels]) if isinstance(labels, list) else set()
    has_water = any(w in labs for w in WATER_LABELS) or any("water" in l for l in labs)
    has_urban = any(u in labs for u in URBAN_LABELS) or any("urban" in l or "industrial" in l for l in labs)
    if has_water:
        extra += " SAR backscatter confirms water presence via low return."
    if has_urban:
        extra += " SAR shows characteristic double-bounce returns consistent with built-up structures."
    # Clearly comment: heuristic template for training signal, not authoritative physics.
    return (base_caption + extra).strip()

def build_synthetic_fusion_dataset(n, seed=42):
    """Fallback that always works — random optical+SAR chips + templated fusion captions."""
    rng = np.random.default_rng(seed)
    rows = []
    for i in range(n):
        arr_opt = rng.integers(0, 255, (224,224,3), dtype=np.uint8)
        if i % 3 == 0:
            arr_opt[:,:,1] = np.clip(arr_opt[:,:,1].astype(int) + 30, 0, 255).astype(np.uint8)
        img_opt = Image.fromarray(arr_opt)
        arr_sar = rng.integers(0, 255, (224,224,3), dtype=np.uint8)
        img_sar = Image.fromarray(arr_sar)
        label = random.choice(LAND_COVER_LABELS)
        base = label_to_caption([label])
        answer = build_answer_with_sar_heuristic([label], base)
        rows.append({"optical_image": img_opt, "sar_image": img_sar, "question": FUSION_QUESTION, "answer": answer, "labels": [label], "image_id": f"syn_fus_{i:05d}"})
    return Dataset.from_list(rows)

# Try to load cached subset from Drive
def load_or_build_dataset(cfg, cache_dir: Path):
    # HF load_from_disk expects the folder itself
    if (cache_dir / "dataset_info.json").exists() or (cache_dir / "state.json").exists() or any(cache_dir.glob("*.arrow")) or (cache_dir / "dataset.arrow").exists():
        try:
            print(f"Found cached dataset at {cache_dir} — loading from disk...")
            ds = load_from_disk(str(cache_dir))
            if isinstance(ds, DatasetDict):
                print(f"Loaded DatasetDict: { {k: len(v) for k,v in ds.items()} }")
                return ds
            print(f"Loaded Dataset: {len(ds)} rows")
            if len(ds) >= cfg.subset_size:
                ds = ds.train_test_split(test_size=cfg.val_ratio, seed=cfg.seed)
                return ds
            return DatasetDict({"train": ds})
        except Exception as e:
            print(f"Cache load failed ({e}) — rebuilding...")

    print(f"No cache at {cache_dir} — building subset (n={cfg.subset_size}) via streaming GFM-Bench/BigEarthNet...")
    try:
        # Streaming same as stage1: GFM-Bench/BigEarthNet has BOTH optical and radar fields, co-registered
        ds_stream = load_dataset("GFM-Bench/BigEarthNet", split="train", streaming=True, trust_remote_code=True)
        ds_stream = ds_stream.shuffle(seed=cfg.seed, buffer_size=1000)
        rows = []
        for idx, ex in enumerate(ds_stream):
            if len(rows) >= cfg.subset_size:
                break
            try:
                # Pull BOTH fields per sample
                optical = ex.get("optical") or ex.get("optical_image") or ex.get("sentinel2") or ex.get("image")
                radar = ex.get("radar") or ex.get("sar") or ex.get("sentinel1") or ex.get("s1")
                labels = ex.get("labels") or ex.get("label") or ex.get("bigearthnet_labels") or []
                if isinstance(labels, int):
                    labels = [str(labels)]
                if optical is None or radar is None:
                    continue
                img_opt = optical_to_pil(optical)
                img_sar = sar_to_pil(radar, mode=cfg.sar_viz_mode)
                base = label_to_caption(labels if isinstance(labels, list) else [str(labels)])
                answer = build_answer_with_sar_heuristic(labels if isinstance(labels, list) else [str(labels)], base)
                rows.append({"optical_image": img_opt, "sar_image": img_sar, "question": FUSION_QUESTION, "answer": answer, "labels": labels, "image_id": ex.get("image_id", f"gfm_{idx:05d}")})
            except Exception as ie:
                # Skip malformed samples, continue streaming
                continue
        if len(rows) < cfg.subset_size * 0.5:
            raise RuntimeError(f"Only collected {len(rows)} samples — too few, falling back to synthetic")
        ds_full = Dataset.from_list(rows)
        print(f"Collected {len(rows)} real GFM-Bench samples with optical+SAR")
    except Exception as e:
        print(f"Real streaming failed ({e}) — falling back to synthetic placeholder pairs (network hiccup tolerated)")
        ds_full = build_synthetic_fusion_dataset(cfg.subset_size, seed=cfg.seed)
    ds_split = ds_full.train_test_split(test_size=cfg.val_ratio, seed=cfg.seed)
    # Save to Drive for next session
    try:
        ds_split.save_to_disk(str(cache_dir))
        print(f"Saved subset cache to {cache_dir} (reuse next session, no rebuild)")
    except Exception as e:
        print(f"Could not save cache (Drive full/permission?): {e}")
    return ds_split

datasetDict = load_or_build_dataset(CFG, DATASET_CACHE_DIR)
if isinstance(datasetDict, Dataset):
    datasetDict = DatasetDict({"train": datasetDict})
print(datasetDict)
# Example: show both modalities
ex0 = datasetDict["train"][0]
print({k: (v if not isinstance(v, Image.Image) else f"<PIL {v.size} {v.mode}>") for k,v in ex0.items() if k in ["optical_image","sar_image","question","answer","labels"]})
try:
    display(ex0["optical_image"].resize((224,224)))
    display(ex0["sar_image"].resize((224,224)))
except Exception:
    pass
print("Q:", ex0["question"])
print("A:", ex0["answer"])


## 7 — Load quantized base model + processor (Qwen2-VL-2B)

`Qwen2VLForConditionalGeneration` with `BitsAndBytesConfig` (NF4, double-quant, fp16 compute). Uses `device_map="auto"` so it shards to GPU.

- Stage 4 continues from stage 3 adapter `imadityasarkar/cdvqa_change` — the LoRA will be loaded on top of this base in §8.
- Dynamic resolution is capped via `min_pixels`/`max_pixels` to keep VRAM flat on T4.


In [ ]:
import gc
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

# Free any leftover memory before the biggest allocation in the notebook
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Loading base: {CFG.base_model}")
processor = AutoProcessor.from_pretrained(
    CFG.base_model,
    min_pixels=CFG.image_min_pixels,
    max_pixels=CFG.image_max_pixels,
    trust_remote_code=True,
)
print(f"Processor: min_pixels={CFG.image_min_pixels} max_pixels={CFG.image_max_pixels}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,  # fp16 on T4
    bnb_4bit_use_double_quant=True,
)

# low_cpu_mem_usage=True + torch_dtype avoid materializing the full fp32
# checkpoint in system RAM before quantizing — this is the single biggest
# RAM spike in the whole notebook, and the most likely cause of a silent
# kernel-restart crash with no Python traceback (Linux OOM killer).
model = Qwen2VLForConditionalGeneration.from_pretrained(
    CFG.base_model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
)
from peft import prepare_model_for_kbit_training
# Gradient checkpointing was turned off in earlier stages due to T4 VRAM/cache interactions;
# we keep it enabled via TrainingArguments but disable prepare_model_for_kbit_training's own checkpointing here if needed.
# Keep existing fix: low_cpu_mem_usage=True already set above; gradient checkpointing controlled in §10.
model = prepare_model_for_kbit_training(model)
print("Base model loaded in 4-bit + prepared for k-bit training.")
print(f"Model device map: {getattr(model, 'hf_device_map', 'auto')}")
if torch.cuda.is_available():
    print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

gc.collect()


## 8 — Configure LoRA (QLoRA)

Rank 16 / alpha 32 (scaling 2.0) is the standard for 2B VLMs on T4 — ~14M trainable params (~0.7% of 2B). Target modules cover both attention (`q/k/v/o`) and MLP (`gate/up/down`).


In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel

lora_config = LoraConfig(
    r=CFG.lora_r,
    lora_alpha=CFG.lora_alpha,
    lora_dropout=CFG.lora_dropout,
    target_modules=list(CFG.lora_target_modules),
    bias="none",
    task_type="CAUSAL_LM",
)
print(lora_config)

# Stage 4 continues from stage 3 adapter (latest in chain: stage1→stage2→stage3).
# Check actual chain in repo: stage1 BigEarthNet -> stage2 VRSBench (imadityasarkar/satquery-qwen2vl-stage1-bigearthnet -> imadityasarkar/satquery-phase2-vrsbench) -> stage3 CDVQA (imadityasarkar/cdvqa_change).
# We load that adapter as starting point via PeftModel.from_pretrained(base, PREVIOUS_ADAPTER_PATH), then continue LoRA training — not raw base.
prev_adapter = CFG.adapter_to_continue
# Try Drive path first (if you have it cached), then HF Hub id
if prev_adapter and Path(prev_adapter).exists():
    print(f"Continuing from local adapter: {prev_adapter}")
    model = PeftModel.from_pretrained(model, prev_adapter, is_trainable=True)
    print("Loaded previous adapter from local Drive path — will be further fine-tuned.")
elif prev_adapter:
    # Try HF Hub id (e.g. imadityasarkar/cdvqa_change)
    try:
        print(f"Continuing from HF adapter: {prev_adapter}")
        model = PeftModel.from_pretrained(model, prev_adapter, is_trainable=True)
        print("Loaded previous adapter from Hub — will be further fine-tuned.")
    except Exception as e:
        print(f"Could not load previous adapter from Hub ({e}) — starting fresh LoRA")
        model = get_peft_model(model, lora_config)
else:
    model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
# Sanity: should be ~0.7% trainable


## 9 — Preprocess dataset for Vision-Language SFT (fusion — two images)

We format each sample as a Qwen chat with **BOTH** images in one user turn (image, image, text — in that order). The processor's `apply_chat_template` produces the correct image placeholders + tokenization.

Labels are the assistant answer only (prompt masked with `-100`). `max_seq_length` truncation keeps VRAM flat. Reuses existing `VisionDataCollator` with `torch.cat` (not `torch.stack`) for `pixel_values`/`image_grid_thw` — this repo already hit and fixed that bug once.


In [ ]:
import copy

def format_sample(example, processor, max_length=CFG.max_seq_length):
    """Return tokenized inputs with labels masked for the user prompt — fusion with TWO images in one turn."""
    # Qwen2-VL supports multiple images natively — one prompt containing both optical AND SAR together
    messages = [
        {"role": "user", "content": [
            {"type": "image", "image": example["optical_image"]},
            {"type": "image", "image": example["sar_image"]},
            {"type": "text", "text": example["question"]}
        ]},
        {"role": "assistant", "content": [
            {"type": "text", "text": example["answer"]}
        ]}
    ]
    full_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    prompt_messages = messages[:1]
    prompt_text = processor.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)

    # Pass BOTH images to processor (optical first, SAR second)
    full = processor(text=[full_text], images=[example["optical_image"], example["sar_image"]], padding=False, return_tensors=None)
    prompt = processor(text=[prompt_text], images=[example["optical_image"], example["sar_image"]], padding=False, return_tensors=None)

    input_ids = full["input_ids"][0]
    labels = copy.deepcopy(input_ids)
    prompt_len = len(prompt["input_ids"][0])
    labels[:prompt_len] = [-100] * prompt_len

    if len(input_ids) > max_length:
        input_ids = input_ids[-max_length:]
        labels = labels[-max_length:]

    out = {
        "input_ids": input_ids,
        "attention_mask": [1]*len(input_ids),
        "labels": labels,
    }
    if "pixel_values" in full:
        out["pixel_values"] = full["pixel_values"]
    if "image_grid_thw" in full:
        out["image_grid_thw"] = full["image_grid_thw"]
    return out

# Quick test on one sample
sample = datasetDict["train"][0]
tok = format_sample(sample, processor)
print(f"input_ids len: {len(tok['input_ids'])} | labels non-masked: {sum(1 for x in tok['labels'] if x!=-100)}")
print(f"Has pixel_values: {'pixel_values' in tok} | image_grid_thw: {'image_grid_thw' in tok}")
# Show that two image grids are present (optical+SAR)
if "image_grid_thw" in tok:
    print(f"image_grid_thw shape: {tok['image_grid_thw'].shape if hasattr(tok['image_grid_thw'], 'shape') else tok['image_grid_thw']}")
print(processor.decode([x for x in tok["input_ids"] if x!=-100][:150]))

def map_fn(example):
    return format_sample(example, processor)

print(f"Tokenizing train/val (n={CFG.subset_size})...")
train_ds = datasetDict["train"].map(
    map_fn, remove_columns=datasetDict["train"].column_names, desc="tokenize train",
    writer_batch_size=50,
)
eval_ds = None
if "test" in datasetDict:
    eval_ds = datasetDict["test"].map(map_fn, remove_columns=datasetDict["test"].column_names, desc="tokenize val", writer_batch_size=50)
elif "validation" in datasetDict:
    eval_ds = datasetDict["validation"].map(map_fn, remove_columns=datasetDict["validation"].column_names, desc="tokenize val", writer_batch_size=50)
else:
    eval_ds = datasetDict["test"].map(map_fn, remove_columns=datasetDict["test"].column_names, desc="tokenize val", writer_batch_size=50) if "test" in datasetDict else None

print(f"Train: {len(train_ds)} | Eval: {len(eval_ds) if eval_ds else 0}")

import gc
del datasetDict
gc.collect()
print("Freed raw image dataset from RAM after tokenization.")


## 10 — Training arguments + resume logic

- `save_steps=25` → frequent Drive saves (tolerates disconnects).
- `resume_from_checkpoint` is auto-detected: the latest `checkpoint-*` under `CHECKPOINT_DIR` is reused on every rerun.
- `fp16=True` (not bf16) for T4. `optim=paged_adamw_8bit` saves ~2GB. `gradient_checkpointing=True` trades ~20% speed for ~4GB VRAM.
- `report_to="none"` by default; set to `trackio`/`wandb` if you want experiment tracking.


In [ ]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq
import glob, os

def find_latest_checkpoint(checkpoint_dir: Path):
    ckpts = sorted(glob.glob(str(checkpoint_dir / "checkpoint-*")), key=lambda p: int(p.split("-")[-1]) if p.split("-")[-1].isdigit() else -1)
    return ckpts[-1] if ckpts else None

latest_ckpt = find_latest_checkpoint(CHECKPOINT_DIR)
print(f"Latest checkpoint: {latest_ckpt if latest_ckpt else '(none — fresh start)'}")

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=CFG.per_device_train_batch_size,
    per_device_eval_batch_size=CFG.per_device_train_batch_size,
    gradient_accumulation_steps=CFG.gradient_accumulation_steps,
    learning_rate=CFG.learning_rate,
    lr_scheduler_type=CFG.lr_scheduler_type,
    warmup_ratio=CFG.warmup_ratio,
    num_train_epochs=CFG.num_train_epochs,
    max_steps=CFG.max_steps if CFG.max_steps>0 else -1,
    weight_decay=CFG.weight_decay,
    optim=CFG.optim,
    max_grad_norm=CFG.max_grad_norm,
    fp16=True,
    bf16=False,
    gradient_checkpointing=CFG.gradient_checkpointing,
    ddp_find_unused_parameters=False,
    logging_steps=CFG.logging_steps,
    save_steps=CFG.save_steps,
    save_total_limit=CFG.save_total_limit,
    save_strategy="steps",
    evaluation_strategy="steps" if eval_ds is not None else "no",
    eval_steps=CFG.eval_steps if eval_ds is not None else None,
    load_best_model_at_end=False,
    report_to="none",
    seed=CFG.seed,
    remove_unused_columns=False,
)
print(training_args)

# Data collator — reuse existing VisionDataCollator with torch.cat (not torch.stack) for vision tensors.
# This repo already hit and fixed the stack vs cat bug once — keep cat.
from transformers import Qwen2VLProcessor
data_collator = DataCollatorForSeq2Seq(
    tokenizer=processor.tokenizer,
    padding=True,
    pad_to_multiple_of=8,
    return_tensors="pt",
)
class VisionDataCollator:
    def __init__(self, base):
        self.base = base
    def __call__(self, features):
        has_pixel = "pixel_values" in features[0]
        pixel_values = [f.pop("pixel_values", None) for f in features]
        image_grid_thw = [f.pop("image_grid_thw", None) for f in features]
        batch = self.base(features)
        if has_pixel and pixel_values[0] is not None:
            import torch as _t
            try:
                # Qwen2-VL: pixel_values per sample is (num_patches*14*14). Use cat (not stack) along dim 0
                # because samples have different num_patches (dynamic resolution).
                tensors = [_t.tensor(p) if not isinstance(p, _t.Tensor) else p for p in pixel_values]
                # cat along first dim keeps variable image sizes; stack would fail on mismatched shapes
                batch["pixel_values"] = _t.cat(tensors, dim=0) if len(tensors)>1 else tensors[0]
            except Exception:
                batch["pixel_values"] = pixel_values
            try:
                thws = [_t.tensor(g) if not isinstance(g, _t.Tensor) else g for g in image_grid_thw]
                batch["image_grid_thw"] = _t.cat(thws, dim=0) if len(thws)>1 else thws[0]
            except Exception:
                batch["image_grid_thw"] = _t.tensor(image_grid_thw) if image_grid_thw[0] is not None else None
        return batch

collator = VisionDataCollator(data_collator)
print("TrainingArguments + collator ready.")
print(f"Effective batch size: {CFG.per_device_train_batch_size * CFG.gradient_accumulation_steps}")


## 11 — Trainer

Plain `transformers.Trainer` (not `TRL SFTTrainer`) for maximum version stability on Colab. The resume logic from §10 is passed here.


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
    tokenizer=processor.tokenizer,
)
print("Trainer created.")
print(f"Train samples: {len(train_ds)} | Eval: {len(eval_ds) if eval_ds else 0}")
print(f"Max steps: {training_args.max_steps if training_args.max_steps>0 else 'epochs=' + str(CFG.num_train_epochs)}")


## 12 — Train (resume-aware, Drive-backed)

This cell is safe to rerun after any disconnect — it automatically resumes from `latest_ckpt` if found.

- First run: `trainer.train()` from scratch, saving to `CHECKPOINT_DIR/checkpoint-25`, `checkpoint-50`, ...
- After disconnect: re-run notebook top-to-bottom; this cell finds `latest_ckpt` and continues.
- On `KeyboardInterrupt` / Colab timeout, the last `save_steps` checkpoint is already on Drive.


In [ ]:
# Train — resume if checkpoint exists
try:
    if latest_ckpt and Path(latest_ckpt).exists():
        print(f"Resuming from {latest_ckpt}")
        trainer.train(resume_from_checkpoint=latest_ckpt)
    else:
        print("Starting fresh training...")
        trainer.train()
except Exception as e:
    print(f"Training interrupted: {e}")
    try:
        trainer.save_state()
        print("Saved trainer state after interruption.")
    except Exception as se:
        print(f"Could not save state: {se}")
    raise

print("Training done.")
print(f"Final checkpoint dir: {CHECKPOINT_DIR}")
!ls -lh "{CHECKPOINT_DIR}" 2>&1 | head -n 50
if torch.cuda.is_available():
    print(f"Peak GPU memory: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB")


## 13 — Save final LoRA adapter to Drive (and optionally Hub)

Only the adapter (~30–80 MB) is saved, not the full 2B base. This adapter path is what `backend/config.py` (new `SATQUERY_FUSION_ADAPTER_PATH`) and future notebooks will load.

- The adapter is saved to `ADAPTER_OUTPUT_DIR` on Drive (`stage4_optical_sar_fusion/final_adapter`) and also to `training/adapters/` locally if present.
- To push to Hugging Face Hub: set `PUSH_TO_HUB = True` and fill `HF_REPO_ID` to a NEW repo (do not overwrite existing stage1-3 repos).


In [ ]:
PUSH_TO_HUB = False
HF_REPO_ID = ""  # e.g. "your-hf-username/satquery-qwen2vl-stage4-optical-sar-fusion" — NEW repo, separate from stage1-3

print(f"Saving adapter to {ADAPTER_OUTPUT_DIR}")
ADAPTER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(ADAPTER_OUTPUT_DIR))
processor.tokenizer.save_pretrained(str(ADAPTER_OUTPUT_DIR))
try:
    processor.save_pretrained(str(ADAPTER_OUTPUT_DIR))
except Exception as e:
    print(f"processor.save_pretrained skipped: {e}")
print(f"Saved adapter files:")
!ls -lh "{ADAPTER_OUTPUT_DIR}" 2>&1 | head -n 30

try:
    local_adapter = Path("training/adapters/stage4_optical_sar_fusion")
    local_adapter.mkdir(parents=True, exist_ok=True)
    (local_adapter / "DRIVE_PATH.txt").write_text(str(ADAPTER_OUTPUT_DIR))
    print(f"Wrote Drive pointer to {local_adapter / 'DRIVE_PATH.txt'}")
except Exception as e:
    print(f"Local pointer skipped: {e}")

if PUSH_TO_HUB and HF_REPO_ID:
    from huggingface_hub import HfApi
    api = HfApi()
    print(f"Pushing adapter to Hub: {HF_REPO_ID}")
    model.push_to_hub(HF_REPO_ID)
    processor.push_to_hub(HF_REPO_ID)
    print("Pushed to Hub — set this repo id in backend/config.py SATQUERY_FUSION_ADAPTER_PATH (separate follow-up task)")
else:
    print("Hub push skipped (set PUSH_TO_HUB=True to enable).")
    print(f"\nFor backend inference (separate task), set in backend/config.py:")
    print(f"  FUSION_ADAPTER_PATH = \"{ADAPTER_OUTPUT_DIR}\"  # or HF_REPO_ID if pushed")
    print(f"  # Do NOT overwrite stage1-3 adapter repos")


## 14 — Sanity-check inference (fusion — optical + SAR)

Loads the freshly saved adapter on top of the 4-bit base and runs the fusion query on a held-out optical+SAR pair. No extra training — just verification that the adapter loads for future `backend/models/fusion.py` wiring.

If this fails with OOM, reduce `max_new_tokens` or restart runtime (model is still on GPU from training).


In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel
from pathlib import Path

import gc
for _name in ["trainer", "model", "train_ds", "eval_ds", "collator"]:
    if _name in globals():
        try:
            del globals()[_name]
        except Exception:
            pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Cleared training model + dataset objects from memory.")

adapter_path = str(ADAPTER_OUTPUT_DIR)
base_id = CFG.base_model
print(f"Loading base {base_id} for inference...")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
base = Qwen2VLForConditionalGeneration.from_pretrained(base_id, quantization_config=bnb, device_map="auto", trust_remote_code=True)
proc = AutoProcessor.from_pretrained(base_id, min_pixels=CFG.image_min_pixels, max_pixels=CFG.image_max_pixels, trust_remote_code=True)
print(f"Loading adapter {adapter_path}...")
peft_model = PeftModel.from_pretrained(base, adapter_path)
peft_model.eval()
print("Adapter loaded.")

# Pick a held-out pair — reuse dataset cache loading if del'd, else rebuild synthetic one-sample
try:
    # If datasetDict was deleted, reload small synthetic for demo
    test_opt = datasetDict["train"][0]["optical_image"] if "datasetDict" in globals() else None
    test_sar = datasetDict["train"][0]["sar_image"] if "datasetDict" in globals() else None
    test_answer = datasetDict["train"][0]["answer"] if "datasetDict" in globals() else ""
    if test_opt is None:
        raise NameError
except Exception:
    # Fallback: build one synthetic pair
    from datasets import load_from_disk
    try:
        ds_disk = load_from_disk(str(DATASET_CACHE_DIR))
        ex = ds_disk["train"][0]
        test_opt = ex["optical_image"]
        test_sar = ex["sar_image"]
        test_answer = ex["answer"]
    except Exception:
        # Last resort synthetic
        test_opt = Image.new("RGB", (224,224), color=(120,180,120))
        test_sar = Image.new("RGB", (224,224), color=(80,80,120))
        test_answer = "Fallback ground truth: forest and water."

question = FUSION_QUESTION
messages = [{"role":"user","content":[{"type":"image","image": test_opt}, {"type":"image","image": test_sar}, {"type":"text","text": question}]}]
text = proc.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = proc(text=[text], images=[test_opt, test_sar], return_tensors="pt", padding=True).to(peft_model.device)
print(f"Prompt: {question}")
print(f"Ground truth (label-derived): {test_answer}")
try:
    display(test_opt.resize((224,224)))
    display(test_sar.resize((224,224)))
except Exception:
    pass

with torch.no_grad():
    out = peft_model.generate(**inputs, max_new_tokens=128, do_sample=False, temperature=0.0)
    gen = out[0][inputs["input_ids"].shape[1]:]
    answer = proc.decode(gen, skip_special_tokens=True)
    print(f"\nModel answer: {answer}")
    print(f"\nGround truth: {test_answer}")
    print("\nSanity check done — compare answer vs ground truth.")


## 15 — Next stages & backend wiring

For **backend**: in a separate follow-up task, wire the new adapter into `backend/models/fusion.py` and `backend/config.py` (`SATQUERY_FUSION_ADAPTER_PATH` env var pointing to this stage4 repo) — do not edit backend in this notebook task.

Checkpoint chain is now: `base` → `imadityasarkar/satquery-qwen2vl-stage1-bigearthnet` → `imadityasarkar/satquery-phase2-vrsbench` → `imadityasarkar/cdvqa_change` → **this stage4** `your-hf-username/satquery-qwen2vl-stage4-optical-sar-fusion`.

Backend inference (`backend/models/` + `backend/registry.py`) will load this stage4 adapter via `PeftModel.from_pretrained(base, adapter_path)` — see `backend/config.py:FUSION_ADAPTER_PATH` after wiring.

---
**Troubleshooting appendix**
- `OutOfMemoryError` → lower `max_seq_length` to 768, or `image_max_pixels` to `384*28*28`, or set `preprocess` truncation earlier.
- `Tokenizer` warnings about `pad_token` → already handled by `DataCollatorForSeq2Seq(pad_to_multiple_of=8)`; safe to ignore.
- `Drive quota exceeded` → delete old `checkpoint-*` folders, keep only last 2 (`save_total_limit=2` does this).
- `Model not found` / 401 → `huggingface-cli login` or set `HF_TOKEN` env if base is gated.
